# 固定描述量的跨快照与跨年份一致性
只比较输入，不读取未来收益；保留所有可用年份、缺失状态和快照差异，不按结果改变特征定义。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from src.data.constituent_descriptors import reported_basket_descriptors
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/constituent_descriptor_stability_20261014';plan=json.loads((card/'plan.json').read_text());qlib.init(provider_uri=plan['provider_uri'],region='us')
def load(name,artifact):
 spec=plan['sources'][name];p=root/spec['card'];s=json.loads((p/'summary.json').read_text());mp=p/'verification_manifest.json';assert hashlib.sha256(mp.read_bytes()).hexdigest()==spec['manifest_sha256'];entry=next(e for e in json.loads(mp.read_text())['entries'] if e['path'].endswith('/artifacts/'+artifact))
 with R.uri_context(s['recorder_uri']):
  rec=R.get_recorder(recorder_id=spec['recorder_id'],experiment_name=p.name);path=Path(rec.download_artifact(artifact));assert hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256'];return rec.load_object(artifact)
old_prices=load('old','analysis/prices');fresh_prices=load('fresh','analysis/prices')['$adj_close'].unstack('instrument');holdings=load('fresh','inputs/holdings')
features={name:reported_basket_descriptors(prices,holdings) for name,prices in [('old',old_prices),('fresh',fresh_prices)]}
parts=[]
for name,values in features.items():
 df=values.copy();df.index=pd.MultiIndex.from_arrays([df.index,[name]*len(df)],names=['datetime','instrument']);parts.append(df)
grouped=pd.concat({'feature':pd.concat(parts).sort_index()},axis=1);ds=DatasetH(handler=DataHandlerLP.from_df(grouped),segments={'all':(None,None)});observed=ds.prepare('all',col_set='feature');assert len(observed)==sum(len(f) for f in features.values())
columns=['new_low_level20','new_low_price_contraction5','downside_top5_share'];overlap=features['old'].index.intersection(features['fresh'].index);comparisons=[]
for col in columns:
 left=features['old'].loc[overlap,col];right=features['fresh'].loc[overlap,col];valid=left.notna()&right.notna();delta=(left-right).abs()
 comparisons.append({'feature':col,'overlap_sessions':len(overlap),'paired_known':int(valid.sum()),'old_known_new_unknown':int((left.notna()&right.isna()).sum()),'old_unknown_new_known':int((left.isna()&right.notna()).sum()),'maximum_absolute_difference':float(delta.loc[valid].max()),'median_absolute_difference':float(delta.loc[valid].median()),'differences_above_1e_minus5':int(delta.loc[valid].gt(1e-5).sum())})
rows=[]
for source,values in features.items():
 for year,part in values.groupby(values.index.year):
  if year<2022:continue
  for col in columns:
   v=part[col].dropna();rows.append({'source':source,'year':year,'feature':col,'calendar_rows':len(part),'known_rows':len(v),'minimum':float(v.min()),'median':float(v.median()),'maximum':float(v.max())})
# Compare exact previously defined quarter outputs where both contracts are known.
reference=load('quarter','analysis/candidate_features');new=features['fresh'].reindex(reference.index)
quarter_comparison=[]
for col in columns:
 mask=reference[col].notna()&new[col].notna();np.testing.assert_allclose(reference.loc[mask,col],new.loc[mask,col],atol=1e-12)
 quarter_comparison.append({'feature':col,'previous_known':int(reference[col].notna().sum()),'current_known':int(new[col].notna().sum()),'newly_known_from_pre_quarter_history':int((reference[col].isna()&new[col].notna()).sum())})
summary={'experiment':card.name,'decision':'fixed_descriptor_source_stability_checked_no_prediction_claim','new_fits':0,'new_backtests':0,'outcome_correlations':0,'parameter_searches':0,'objective_achieved':False,'provider_uri':plan['provider_uri'],'region':'us','as_of':'2026-09-30','qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','interpretation':'Input-only cross-vintage and missingness audit. No independent forecast validation; historical windows already reused. Extra known early-quarter changes come from actual pre-quarter observations, not zero fill.','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/features','sources/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='descriptor_stability_without_fit',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0)
 rec.save_objects(artifact_path='inputs',plan=plan)
 rec.save_objects(artifact_path='analysis',features=features,overlap=pd.DataFrame(comparisons),annual=pd.DataFrame(rows),quarter=pd.DataFrame(quarter_comparison),summary=summary)
 rec.save_objects(artifact_path='sources',builder=(root/'src/data/constituent_descriptors.py').read_text())
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');pd.DataFrame(comparisons).to_csv(card/'snapshot_overlap.csv',index=False);pd.DataFrame(rows).to_csv(card/'annual_feature_ranges.csv',index=False);pd.DataFrame(quarter_comparison).to_csv(card/'quarter_warmup.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':plan['provider_uri'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False));print(pd.DataFrame(comparisons).to_string(index=False));print(pd.DataFrame(quarter_comparison).to_string(index=False))


# 固定描述量跨快照核查与回看窗口修复

0拟合、0回测、0未来收益相关检验、0参数搜索。本轮把既定三个描述量统一为可复用构建函数，使用真实完整历史构造后再截取分析季度，不改变20日、5日、前5名定义。

两个价格快照重叠86个交易日。在双方均已知的日期，新低水平67日、五日价格指标收敛62日、下跌前5名集中度79日可比较；最大绝对差分别约1.1e-16、5.6e-16、1.16e-6，未出现大于固定1e-5的差异。只有重叠日期被比较，不能认证非重叠新数据或历史首次接收。

新快照从3月开始，最初19个新低窗口、24个收敛窗口及首个日收益缺少回看，均保持未知；旧快照在这些日期已知不构成数据异常。不同起始日期不能靠零填充强行对齐。

此前Q3单独构造导致首五个收敛值未知。现在利用真实季度前历史和当时可知篮子，Q3收敛已知日由59增至64；此前已知59日逐值一致。新低水平仍64日，下跌集中度仍61日，另外3日没有负贡献，继续保持未知。没有通过补0增加覆盖。

所有可用年份的范围与缺失数均保留在annual_feature_ranges.csv；不同年份或数据窗口的分布变化不被自动称作错误，不据此重新标准化、删极端值或选年份。未来披露不能改变过去特征、无下跌贡献不得造出集中度0的合同测试通过；Qlib Dataset读回及来源哈希、NAS回读通过。

决定：fixed_descriptor_source_stability_checked_no_prediction_claim。这说明字段定义在重叠快照上可重现，不能证明预测性、减少模型过拟合或产生净超额。未接入现役模型，未重开失败广度路线。下一步的任何信息检验都必须冻结这份定义、保留未成熟标签，并明确研究历史已复用；不得继续围绕坏季度微调周期。

[快照重叠](snapshot_overlap.csv) · [季度回看窗口](quarter_warmup.csv) · [全部年份范围](annual_feature_ranges.csv)

Recorder `625e46daece84f01a2228d658dcd1638`。
